In [ ]:
# Lab type: review
# Course: AI402 — Retrieval & RAG Systems
# Lesson: Agentic and Iterative Retrieval
# Task: A scripted agentic retrieval loop is below (the 'model' is a
# deterministic stand-in, so no API key is needed). Run it and judge the
# orchestration: what is enforced, what is merely hoped for.

# Lab: Reviewing an Agentic Retrieval Loop

The `scripted_model` below plays the LLM's role with a fixed policy so the loop's *orchestration* can be studied deterministically: it answers a multi-hop question by first finding a customer, then searching for that customer's renewal.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

In [ ]:
# Extra docs that make the question multi-hop
AGENT_DOCS = CORPUS + [
    ("ticket-stats", "Support > Quarterly stats",
     "Acme Corp filed 41 support tickets last quarter, the most of any "
     "customer. Globex filed 12."),
    ("acme-renewal", "Contracts > Acme",
     "Acme Corp renewed its Enterprise contract on March 1 for two years."),
]
A_IDS = [d[0] for d in AGENT_DOCS]
A_TEXTS = [f"{d[1]}: {d[2]}" for d in AGENT_DOCS]
A_EMB = embed(A_TEXTS)

def search_kb(query, k=2):
    """The tool: the same funnel every lab has used (dense here)."""
    scores = A_EMB @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(A_IDS[i], A_TEXTS[i], float(scores[i])) for i in order]

QUESTION = "did the customer who filed the most tickets last quarter renew?"

## The loop under review

In [ ]:
# --- THE ORCHESTRATION LOOP (review this code — is it correct?) ---
def scripted_model(question, observations):
    """Deterministic stand-in for the LLM's tool-use decisions."""
    seen = " ".join(t for _, t, _ in observations)
    if "most of any customer" not in seen:
        return ("search", "which customer filed the most support tickets last quarter")
    if "renewed" not in seen:
        return ("search", "did Acme Corp renew its contract")
    return ("answer", "Yes — Acme Corp (most tickets: 41) renewed on March 1.")

def agentic_answer(question):
    observations = []
    while True:                                   # <- look closely
        action, payload = scripted_model(question, observations)
        if action == "answer":
            return payload, observations
        observations += search_kb(payload)         # <- and here

answer, obs = agentic_answer(QUESTION)
print("answer:", answer)
print(f"hops: {len(obs) // 2}, observations: {[d for d, _, _ in obs]}")

**Question 1.** Single-shot retrieval cannot answer QUESTION well — demonstrate it: run `search_kb(QUESTION, k=3)` and explain, from the results, why the second hop's query could not have been written up front.

<details>
<summary>🔑 Reveal answer — Question 1</summary>

Single-shot retrieval finds `ticket-stats` (the question's vocabulary matches it) but has no reason to rank `acme-renewal` highly — the question never mentions Acme. "Acme" only enters the picture after reading the first hop's result; the second query is *derived from the first answer*, which no up-front rewrite or decomposition can know. That dependency is the definition of a multi-hop question.

</details>

**Question 2.** The loop works on this happy path. List what bounds it: iteration cap? token budget? latency budget? What happens if `scripted_model` is replaced by a real LLM that keeps choosing "search" — and whose job is it to prevent that?

<details>
<summary>🔑 Reveal answer — Question 2</summary>

Nothing bounds it: `while True` with no iteration cap, no token budget, no latency budget. A real model that never reaches an "answer" state spins until something external breaks, accumulating context (and cost) each hop. Enforcement belongs to the **orchestrator** — a hard cap on search calls (3–5 typical), per-request token/latency budgets checked in the loop — because the model does not reliably know when to stop. Telling it to "search within reason" is a hope, not a control.

</details>

**Question 3.** Add the guards: rewrite `agentic_answer` with `max_searches=3` and a `token_budget` (estimate tokens as `len(text) // 4` accumulated over observations); on breach, return the honest failure `("budget exhausted", observations)`. Verify the happy path still completes in 2 hops.

In [ ]:
# Work here: bounded agentic_answer with max_searches and token_budget.


<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
def agentic_answer_bounded(question, max_searches=3, token_budget=2000):
    observations, searches, tokens = [], 0, 0
    while True:
        action, payload = scripted_model(question, observations)
        if action == "answer":
            return payload, observations
        if searches >= max_searches or tokens > token_budget:
            return "budget exhausted", observations
        results = search_kb(payload)
        observations += results
        searches += 1
        tokens += sum(len(t) // 4 for _, t, _ in results)

print(agentic_answer_bounded(QUESTION)[0])   # completes in 2 hops
```

The budget check lives in the loop, not in the model's prompt — the orchestrator owns the guarantee.

</details>

**Question 4.** The final answer above is correct. Explain why a final-answer-only eval would still under-specify this system, and what the lesson says a multi-hop eval set must label.

<details>
<summary>🔑 Reveal answer — Question 4</summary>

A correct final answer can be reached by lucky wandering — querying the wrong customer and stumbling onto the right doc scores identically to a sound trajectory. Multi-hop eval sets label the *intermediate* answers (here: hop 1 must identify Acme via `ticket-stats`; hop 2 must retrieve `acme-renewal`), so per-hop retrieval quality, convergence, and stopping behaviour are all measurable — trajectory evaluation on top of answer evaluation.

</details>

## Summary

1. Agentic retrieval enables _______ questions, where the next query depends on the last result.
2. Loop bounds are enforced by the _______, never the model.
3. Trajectory evals need labelled _______ answers, not just final ones.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **multi-hop**
2. **orchestrator**
3. **intermediate**

</details>